In [ ]:
from datetime import datetime
import pandas as pd
from matplotlib import pyplot as plt

from emu_renewal.inputs import OUTPUTS_PATH, DATA_PATH
from emu_renewal.outputs import get_country_analyses

In [ ]:
job_path = OUTPUTS_PATH / "42780561"
countries = [p.parts[-1] for p in job_path.iterdir()]
g_mob = {}
proc_centiles = {}
for country in countries:
    country_path = job_path / country
    analyses = get_country_analyses(country_path)
    proc_samples = {a: pd.read_hdf(country_path / a / "spaghetti.h5")["process"] for a in analyses}
    proc_centiles[country] = proc_samples["no_mob"].quantile([0.05, 0.5, 0.95], axis=1).T
    g_mob[country] = pd.read_csv(DATA_PATH / f"mobility/{country}_gmob_data.csv", index_col=0)
    g_mob[country].index = pd.to_datetime(g_mob[country].index)
    g_mob[country] = g_mob[country].rolling(7, center=True).mean()

In [ ]:
def compare_proc_versus_mobility(proc_centiles, g_mob, mob_type):
    n_rows = 4
    mob_comparison_fig, axes = plt.subplots(n_rows, 4, figsize=[15, 15], sharex=True)
    flat_axes = axes.ravel()
    for c, country in enumerate(countries):
        c_ax = flat_axes[c]
        c_proc_centiles = proc_centiles[country]
        c_ax.plot(c_proc_centiles.index, c_proc_centiles[0.5], label="process", color="navy")
        c_ax.fill_between(c_proc_centiles.index, c_proc_centiles[0.05], c_proc_centiles[0.95], alpha=0.2, color="navy")
        mobility = g_mob[country].loc[g_mob[country].index < c_proc_centiles.index[-1]]
        c_ax.plot(mobility.index, mobility[mob_type], label="mobility")
        c_ax.set_title(country)
        plt.setp(c_ax.xaxis.get_majorticklabels(), rotation=70)
        if c_ax.get_subplotspec().rowspan.stop != n_rows:
            c_ax.set_xticklabels([])
        plt.setp(c_ax.xaxis.get_majorticklabels(), rotation=70)
    mob_comparison_fig.tight_layout()

In [ ]:
compare_proc_versus_mobility(proc_centiles, g_mob, "retail_and_recreation")